# Corruption

# Measure and data collection

Primero tratamos lo general en excel, eliminamos las columnas que no tienen valores, eliminamos las columnas donde todos los valores sean iguales, eliminamos el idioma, la fecha en la cual fue modificado, nos quedamos con la columna FactValueNumeric porque ahi es donde esta el valor que queremos,eliminamos los limites inferior y superior de los intervalos de confianza y nos quedamos con el valor estimado para nuestro estudio. Eliminamos el nombre y codigo del indicador. Y quitamos el parento location code y dejamos nada mas el nombre. Eliminamos Dim1ValueCode.
Cambiamos los nombres de columna a lo siguiente:
FactValueNumeric : suicide_rates
Dim1: sex
Period: year
Location: country
SpatialDimValueCode: country_code
ParentLocation: region

Fist, the libraries

In [1]:
import pandas as pd
from pathlib import Path
import os

Creating the dataframes

For the dataframes that we have not flagged as different, the data were retrieved from worldbank.org. This type of dataframe contains four header rows at the beginning; therefore, the dataframes that come from that source will be read starting from the fifth row.

In [2]:
#Economics
df_gdp = pd.read_csv("economics/gdp.csv", skiprows=4)
df_inflation = pd.read_csv("economics/inflation.csv", skiprows=4)
df_unemp = pd.read_csv("economics/unemp.csv", skiprows=4)
#Inequality
df_gini = pd.read_csv("inequality/gini.csv", skiprows=4)
df_poverty = pd.read_csv("inequality/poverty.csv", skiprows=4)
#Laborhood
df_labor = pd.read_csv("labor/labor.csv", skiprows=4)
df_informalemp = pd.read_csv("labor/informalemp.csv") #This dataframe must be treated differently since it comes from a different source than the others.
#Education
df_literacy = pd.read_csv("education/literacy.csv", skiprows=4)
df_tertiaryeduc = pd.read_csv("education/tertiaryeduc.csv", skiprows=4)
#Health
df_alcohol = pd.read_csv("health/alcohol.csv") #This dataframe must be treated differently since it comes from a different source than the others.
df_depression = pd.read_csv("health/depression.csv") #This dataframe must be treated differently since it comes from a different source than the others.
#Governance
df_governance = pd.read_csv("governance/government.csv", skiprows=4) 
df_homicides = pd.read_csv("governance/homicides.csv", skiprows=4) 

The data for the following corruption perception dataframe were not found as a single dataset but rather separated by year; therefore, we will proceed to merge them all together.
We are going to separate the files which have a year name format (YYYY).csv using a loop and finally merge them with the one with different format.

In [3]:
#This is the file that has a different format from the another corruption files
df_corruption = pd.read_csv("governance/corruption perception/corruption_perception.csv")

In [4]:
folder_path = "governance/corruption perception"
files = os.listdir(folder_path)
#Filtering only year named files YYYY.csv
year_files = [
    f for f in files
    if f.endswith(".csv") and f[:-4].isdigit()
]
year_files = sorted(year_files)

print("Year files found:", len(year_files))
print("Example:", year_files[:10])

Year files found: 12
Example: ['2000.csv', '2001.csv', '2002.csv', '2003.csv', '2004.csv', '2005.csv', '2006.csv', '2007.csv', '2008.csv', '2009.csv']


In [5]:
#Reading each year file, and adding the year column from the filename, then storing into a list
dfs = []

for file in year_files:
    year = int(file[:-4])  # "2000.csv" -> 2000
    
    df = pd.read_csv(os.path.join(folder_path, file))
    df["year"] = year      # THIS is how we know which rows belong to which year
    
    dfs.append(df)

print("DataFrames loaded:", len(dfs))

DataFrames loaded: 12


In [6]:
#Combining all years into one DataFrame
if len(dfs) == 0:
    raise ValueError("No yearly CSV files were loaded. Check folder_path and filenames.")

combined_df = pd.concat(dfs, ignore_index=True)


In [7]:
#Quick validation: each year should have rows (often the same count each year)
print("Combined shape:", combined_df.shape)
print(combined_df["year"].value_counts().sort_index())

Combined shape: (1785, 8)
year
2000     90
2001     91
2002    102
2003    133
2004    146
2005    159
2006    163
2007    180
2008    180
2009    180
2010    178
2011    183
Name: count, dtype: int64


In [8]:
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1785 entries, 0 to 1784
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   country   1785 non-null   str   
 1   iso       1785 non-null   str   
 2   region    1785 non-null   str   
 3   score     1785 non-null   object
 4   rank      1785 non-null   int64 
 5   interval  1626 non-null   object
 6   year      1785 non-null   int64 
 7   range     159 non-null    str   
dtypes: int64(2), object(2), str(4)
memory usage: 138.6+ KB


In [9]:
df_corruption.head()

,country,country_code,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012
0,Denmark,DNK,88.0,88.0,87.0,88.0,88.0,90.0,91.0,92.0,91.0,90.0
1,New Zealand,NZL,88.0,88.0,87.0,87.0,89.0,90.0,91.0,91.0,91.0,90.0
2,Finland,FIN,88.0,85.0,86.0,85.0,85.0,89.0,90.0,89.0,89.0,90.0
3,Singapore,SGP,85.0,85.0,85.0,85.0,84.0,84.0,85.0,84.0,86.0,87.0
4,Sweden,SWE,85.0,85.0,85.0,85.0,84.0,88.0,89.0,87.0,89.0,88.0


In [10]:
print("combined_df countries:", combined_df["country"].nunique())
print("df_corruption countries:", df_corruption["country"].nunique())

combined_df countries: 226
df_corruption countries: 181


### Combining the corruption dataframes

We have to reshape the corruption df. From wide to long format

In [11]:
df_corruption_long = df_corruption.melt(
    id_vars=["country", "country_code"],
    var_name="year",
    value_name="score"
)

Change the year data type. From string to date

In [12]:
df_corruption_long["year"] = df_corruption_long["year"].astype(int)

In [13]:
df_corruption_long.head()

,country,country_code,year,score
0,Denmark,DNK,2021,88.0
1,New Zealand,NZL,2021,88.0
2,Finland,FIN,2021,88.0
3,Singapore,SGP,2021,85.0
4,Sweden,SWE,2021,85.0


In [14]:
df_corruption_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 1810 entries, 0 to 1809
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       1810 non-null   str    
 1   country_code  1810 non-null   str    
 2   year          1810 non-null   int64  
 3   score         1769 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 77.0 KB


Cleaning the countrynames

In [15]:
combined_df["country"] = combined_df["country"].str.strip()
df_corruption_long["country"] = df_corruption_long["country"].str.strip()

Merging the datasets vertically.

In [16]:
corruption_full = pd.concat(
    [
        combined_df[["country", "iso", "region", "score", "year"]],
        df_corruption_long.rename(columns={"country_code": "iso"})[["country", "iso", "score", "year"]]
    ],
    ignore_index=True
)

In [17]:
corruption_full["country"].nunique()

230

Reviewing whether there are inconsistencies in the country names within our dataframe.

In [18]:
unique_countries = sorted(corruption_full["country"].unique())

print(unique_countries)

['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia', 'Bosnia & Herzegovina', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Congo Democratic Republic', 'Congo Republic', 'Congo, Democratic Republic', 'Congo, Republic', 'Congo, Republic of', 'Congo, Republic of the', 'Congo-Brazzaville', 'Congo. Democratic Republic', 'Congo. Republic of', 'Costa Rica', "Cote d'Ivoire", 'Cote d´Ivoire', 'Côte d ́Ivoire', 'Croatia', 'Cuba', 'Cyprus', 'Czech Republic', 'Czech Republik', 'Czechia', 'Côte d´Ivoire', "Côte-d'Ivoire", 'Democratic Republic of Congo', 'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Domi

In [19]:
import re
import unicodedata

def _basic_normalize(s: str) -> str:
    """
    Fix the most common 'invisible' problems:
    - leading/trailing spaces
    - different apostrophes/accents (unicode)
    - multiple spaces
    """
    if pd.isna(s):
        return s

    s = str(s).strip()

    # Normalize unicode (turn weird accents into consistent form)
    s = unicodedata.normalize("NFKC", s)

    # Normalize apostrophes/quotes
    s = s.replace("’", "'").replace("`", "'").replace("´", "'")

    # Remove weird extra spaces
    s = re.sub(r"\s+", " ", s)

    return s


# Canonical names chosen to be "World Bank style" when possible (helps merges later):
# - Congo, Dem. Rep. / Congo, Rep.
# - Korea, Rep. / Korea, Dem. People's Rep.
# - Czechia, Cabo Verde, Eswatini, Viet Nam, etc.
COUNTRY_FIX = {
    # Bosnia
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",

    # Brunei
    "Brunei": "Brunei Darussalam",

    # Cape Verde
    "Cape Verde": "Cabo Verde",

    # Congo (DRC)
    "Congo Democratic Republic": "Congo, Dem. Rep.",
    "Congo, Democratic Republic": "Congo, Dem. Rep.",
    "Congo. Democratic Republic": "Congo, Dem. Rep.",
    "Democratic Republic of Congo": "Congo, Dem. Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",

    # Congo (Republic)
    "Congo Republic": "Congo, Rep.",
    "Congo, Republic": "Congo, Rep.",
    "Congo, Republic of": "Congo, Rep.",
    "Congo, Republic of the": "Congo, Rep.",
    "Congo-Brazzaville": "Congo, Rep.",
    "Congo. Republic of": "Congo, Rep.",

    # Ambiguous "Congo" (you can change this if you prefer DRC)
    "Congo": "Congo, Rep.",

    # Côte d'Ivoire (many encodings)
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Cote d´Ivoire": "Côte d'Ivoire",
    "Côte d ́Ivoire": "Côte d'Ivoire",
    "Côte d´Ivoire": "Côte d'Ivoire",
    "Côte-d'Ivoire": "Côte d'Ivoire",

    # Czech
    "Czech Republic": "Czechia",
    "Czech Republik": "Czechia",

    # Dominican Republic
    "Dominican Rep": "Dominican Republic",
    "Dominican Rep.": "Dominican Republic",

    # Guinea-Bissau
    "Guinea Bissau": "Guinea-Bissau",

    # Korea
    "Korea (North)": "Korea, Dem. People's Rep.",
    "Korea, North": "Korea, Dem. People's Rep.",
    "Korea (South)": "Korea, Rep.",
    "Korea, South": "Korea, Rep.",
    "South Korea": "Korea, Rep.",

    # Kiribati (typo)
    "Kribati": "Kiribati",

    # Kuwait (typo)
    "Kuweit": "Kuwait",

    # Macao/Macau
    "Macau": "Macao",

    # North Macedonia
    "FYR Macedonia": "North Macedonia",
    "Macedonia": "North Macedonia",

    # Moldova (typo)
    "Moldovaa": "Moldova",

    # São Tomé and Príncipe
    "Sao Tome & Principe": "Sao Tome and Principe",

    # Serbia & Montenegro
    "Serbia & Montenegro": "Serbia and Montenegro",

    # Slovak Republic
    "Slovak Republic": "Slovakia",

    # Swaziland / Eswatini
    "Swaziland": "Eswatini",

    # Trinidad & Tobago
    "Trinidad & Tobago": "Trinidad and Tobago",

    # USA variations
    "USA": "United States",
    "United States of America": "United States",

    # Vietnam
    "Vietnam": "Viet Nam",

    # Palestine naming (World Bank commonly uses "West Bank and Gaza")
    "Palestinian Authority": "West Bank and Gaza",
    "Palestine": "West Bank and Gaza",
}


def clean_country_column(df: pd.DataFrame, col: str = "country") -> pd.DataFrame:
    """
    Applies:
    1) basic unicode/space normalization
    2) mapping replacements for known variants in your list
    """
    df = df.copy()
    df[col] = df[col].apply(_basic_normalize)
    df[col] = df[col].replace(COUNTRY_FIX)
    return df


# Example usage:
# corruption_full = clean_country_column(corruption_full, "country")

# Optional: verify what still looks suspicious after cleaning
# (shows any remaining values that still contain commas/parentheses or weird characters)
def show_remaining_suspicious(df: pd.DataFrame, col: str = "country", n: int = 200):
    vals = pd.Series(sorted(df[col].dropna().unique()))
    suspicious = vals[vals.str.contains(r"[().]|  |´|’|`", regex=True)]
    print("Suspicious unique names (after cleaning):")
    print(suspicious.head(n).to_list())


# Optional: check duplicates created by standardization (same country-year twice)
def dup_country_year(df: pd.DataFrame, country_col="country", year_col="year"):
    dups = df[df.duplicated(subset=[country_col, year_col], keep=False)].sort_values([country_col, year_col])
    return dups

In [20]:
#Replacing the name of the country "Cote d’Ivoire" separately.

In [21]:
corruption_full["country"] = corruption_full["country"].replace(
    "Cote d ́Ivoire", "Côte d'Ivoire"
)

In [22]:
corruption_full = clean_country_column(corruption_full, "country")
print(corruption_full["country"].nunique())

191


In [23]:
corruption_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 3595 entries, 0 to 3594
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   country  3595 non-null   str   
 1   iso      3595 non-null   str   
 2   region   1785 non-null   str   
 3   score    3554 non-null   object
 4   year     3595 non-null   int64 
dtypes: int64(1), object(1), str(3)
memory usage: 186.3+ KB


Now we are going to handle the missing values.

In [24]:
corruption_full["score"].isna().sum()

np.int64(41)

In [25]:
corruption_full[corruption_full["score"].isna()].head(20)

,country,iso,region,score,year
1965,Brunei Darussalam,BRN,NaN,NaN,2021
2010,Fiji,FJI,NaN,NaN,2020
2191,Fiji,FJI,NaN,NaN,2019
2372,Fiji,FJI,NaN,NaN,2018
2553,Fiji,FJI,NaN,NaN,2017
2712,Seychelles,SYC,NaN,NaN,2016
2734,Fiji,FJI,NaN,NaN,2016
2757,Vanuatu,VUT,NaN,NaN,2016
2811,Eswatini,SWZ,NaN,NaN,2016
2862,Equatorial Guinea,GNQ,NaN,NaN,2016


In [26]:
corruption_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 3595 entries, 0 to 3594
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   country  3595 non-null   str   
 1   iso      3595 non-null   str   
 2   region   1785 non-null   str   
 3   score    3554 non-null   object
 4   year     3595 non-null   int64 
dtypes: int64(1), object(1), str(3)
memory usage: 186.3+ KB


In [27]:
# Remove the 'region' column
corruption_full = corruption_full.drop(columns=['region'])

# Rename 'iso' column to 'country_code'
corruption_full = corruption_full.rename(columns={'iso': 'country_code'})

# Verify the changes
corruption_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 3595 entries, 0 to 3594
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   country       3595 non-null   str   
 1   country_code  3595 non-null   str   
 2   score         3554 non-null   object
 3   year          3595 non-null   int64 
dtypes: int64(1), object(1), str(2)
memory usage: 152.2+ KB


In [28]:
# Filter rows where 'score' is null
null_scores = corruption_full[corruption_full['score'].isna()]

# Count null values per country
null_count_per_country = null_scores.groupby('country').size()

# Count total records per country
total_per_country = corruption_full.groupby('country').size()

# Calculate percentage of missing values
null_percentage = (null_count_per_country / total_per_country) * 100

# Combine results into a DataFrame
null_summary = pd.DataFrame({
    'null_count': null_count_per_country,
    'total_records': total_per_country,
    'null_percentage': null_percentage
})

# Sort by highest percentage
null_summary = null_summary.sort_values(by='null_percentage', ascending=False)

null_summary.head(20)

,null_count,total_records,null_percentage
country,,,
Fiji,9.0,11,81.818182
Grenada,4.0,12,33.333333
Vanuatu,5.0,15,33.333333
Maldives,4.0,15,26.666667
Solomon Islands,4.0,15,26.666667
Brunei Darussalam,3.0,13,23.076923
Equatorial Guinea,3.0,17,17.647059
Eswatini,2.0,17,11.764706
South Sudan,1.0,10,10.000000


# Data Filtering, Reshaping, and Missing-Value Treatment
## Corruption cleaning
We will clean the missing values in the score variable using a two-step approach to preserve data quality and keep the time-series structure by country.

What we will do and why:

Drop countries with too many missing scores (more than 30%).
Why: When a country is missing a large share of its values (e.g., 30%–80%), imputing would introduce a lot of uncertainty and can bias trends and any downstream modeling.

For the remaining countries (those with mostly complete data), fill the few missing years using linear interpolation within each country over time.
Why: Corruption index scores typically change gradually year-to-year, so interpolating a small number of missing points preserves the country’s trend better than using a global average or arbitrary values.

In [29]:
# Ensure score is numeric (in case it's stored as object)
corruption_full['score'] = pd.to_numeric(corruption_full['score'], errors='coerce')

# 1) Compute missing % per country
null_count_per_country = corruption_full.groupby('country')['score'].apply(lambda x: x.isna().sum())
total_per_country = corruption_full.groupby('country')['score'].size()
null_percentage = (null_count_per_country / total_per_country) * 100

null_summary = pd.DataFrame({
    'null_count': null_count_per_country,
    'total_records': total_per_country,
    'null_percentage': null_percentage
}).sort_values(by='null_percentage', ascending=False)

# 2) Drop countries with > 30% missing
threshold = 30
keep_countries = null_summary[null_summary['null_percentage'] <= threshold].index

corruption_clean = corruption_full[corruption_full['country'].isin(keep_countries)].copy()

# 3) Interpolate remaining missing scores within each country over time
corruption_clean = corruption_clean.sort_values(['country', 'year'])

corruption_clean['score'] = corruption_clean.groupby('country')['score'].transform(
    lambda s: s.interpolate(method='linear', limit_direction='both')
)

# 4) Quick checks
print("Rows before:", len(corruption_full))
print("Rows after :", len(corruption_clean))
print("Remaining missing scores:", corruption_clean['score'].isna().sum())

# Optional: see which countries (if any) still have missing values after interpolation
countries_still_missing = corruption_clean.loc[corruption_clean['score'].isna(), 'country'].unique()
countries_still_missing[:20]

Rows before: 3595
Rows after : 3557
Remaining missing scores: 0


<ArrowStringArray>
[]
Length: 0, dtype: str

In [30]:
corruption_clean

,country,country_code,score,year
678,Afghanistan,AFG,2.5,2005
1055,Afghanistan,AFG,1.8,2007
1064,Afghanistan,AFG,1.5,2008
1422,Afghanistan,AFG,1.3,2009
1599,Afghanistan,AFG,1.4,2010
...,...,...,...,...
2666,Zimbabwe,ZWE,22.0,2017
2485,Zimbabwe,ZWE,22.0,2018
2304,Zimbabwe,ZWE,24.0,2019
2123,Zimbabwe,ZWE,24.0,2020


In [31]:
corruption_clean.to_parquet("cleaned_data/df_corruption.parquet", index=False)